# TB-AWFNO — Burgers 1D: complete temporal-rank sweep

**Full training-size sweep:** N = 25, 50, 100, 200, 400, 800. For K=5, the two integer ranks are M=2 and M=3. This notebook trains **TB-AWFNO only**, once for each (N, M): 12 configurations at seed 1234. It does not rerun FNO, U-Net or direct AWFNO.

Same dataset path as the original paper code, **fixed original train/validation/test partition** across every N, original spatial quadrature and autoregressive rollout, and best-validation checkpoint selection. Outputs go to `MyDrive/AWFNO_TB_rank_sweep/burgers_1d/N.../M.../`. Completed configurations are skipped, and newly trained jobs save intermediate optimizer checkpoints every five epochs (Navier–Stokes uses its original resumable training routine).

**Default budget:** 400 epochs per configuration. The existing completed N=800/M=3 result used 400 epochs and will be reused if its metadata match. Ranks change model capacity; report parameter counts for each comparison. These notebooks have not been GPU-executed here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys, os, json, hashlib, subprocess, urllib.request, time
import torch, pandas as pd, matplotlib.pyplot as plt
assert torch.cuda.is_available(), 'Select GPU in Runtime > Change runtime type.'
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
#@title Full rank + training-size sweep configuration
PDE='burgers_1d'
K=5
TRAIN_POOL_SIZE=800
TRAIN_SIZES=[25,50,100,200,400,800]
EPOCHS=400
SEED=1234
RANKS=sorted(set((K//2,(2*K)//3)))  # exact integer floor
DATA_PATH=Path('/content/drive/MyDrive/anus/1D_Burgers_Sols_Nu0.001.hdf5')
OUTPUT_ROOT=Path('/content/drive/MyDrive/AWFNO_TB_rank_sweep/burgers_1d')
FORCE_RETRAIN=False  # retain completed N/M pairs
assert all(0<n<=TRAIN_POOL_SIZE for n in TRAIN_SIZES)
assert all(1<=m<=K for m in RANKS)
assert DATA_PATH.is_file(),f'Missing dataset: {DATA_PATH}'
OUTPUT_ROOT.mkdir(exist_ok=True,parents=True)
print('PDE:',PDE,'N:',TRAIN_SIZES,'K:',K,'ranks:',RANKS)
print('Total training configurations:',len(TRAIN_SIZES)*len(RANKS))
print('Dataset:',DATA_PATH,'bytes:',DATA_PATH.stat().st_size)


In [ ]:
#@title Retrieve original experiment source and the rank-only runner
SOURCE_URL='https://raw.githubusercontent.com/dimigourz/AWFNO/main/experiments/burgers_1d/ablations.py'
RUNNER_URL='https://raw.githubusercontent.com/dimigourz/AWFNO/main/experiments/tb_rank_runner.py'
SOURCE_FILE=Path('/content/tb_reference_burgers_1d.py')
RUNNER_FILE=Path('/content/tb_rank_runner.py')
def fetch_code(url,target):
    request=urllib.request.Request(url,headers={'Cache-Control':'no-cache','Pragma':'no-cache'})
    content=urllib.request.urlopen(request,timeout=120).read()
    assert len(content)>3000,f'Source download too small: {url}'
    target.write_bytes(content)
    import py_compile
    py_compile.compile(str(target),doraise=True)
    print('Downloaded',target.name,'SHA256',hashlib.sha256(content).hexdigest())
fetch_code(SOURCE_URL,SOURCE_FILE)
fetch_code(RUNNER_URL,RUNNER_FILE)


In [ ]:
#@title Run every N x M configuration; resume safely after disconnects
SOURCE_SHA=hashlib.sha256(SOURCE_FILE.read_bytes()).hexdigest()
DEFAULT_ENV={"ALL_SAMPLE_SIZES":"25,50,100,200,400,800","N_VAL":"50","N_TEST":"100","TARGET_NX":"512","WINDOW_K":"5","WINDOW_STRIDE":"5","BATCH_SIZE":"128","LR":"0.001","WEIGHT_DECAY":"1e-6","FINAL_LOSS_WEIGHT":"1.0","SPLIT_MODE":"contiguous","SPLIT_SEED":"2026","LOCAL_RADIUS":"3"}
for n_train in TRAIN_SIZES:
    for m in RANKS:
        run_dir=OUTPUT_ROOT/f'N{n_train}'/f'M{m}'
        run_dir.mkdir(exist_ok=True,parents=True)
        result_path=run_dir/'result.json'
        if result_path.exists() and not FORCE_RETRAIN:
            saved=json.loads(result_path.read_text())
            identifiers={'pde':PDE,'N_train':n_train,'K':K,'M':m,
                         'seed':SEED,'source_sha256':SOURCE_SHA}
            if all(saved.get(k)==v for k,v in identifiers.items()):
                if 'epochs_configured' not in saved:
                    print('NOTE: reusing historical result without epoch metadata:',result_path)
                elif saved['epochs_configured']!=EPOCHS:
                    raise RuntimeError(f'Existing result has {saved["epochs_configured"]} epochs vs requested {EPOCHS}; set a new output root or FORCE_RETRAIN=True')
                print('[SKIP] Completed N=',n_train,'M=',m,flush=True)
                continue
            raise RuntimeError(f'Existing result has mismatched configuration: {result_path}; use a different OUTPUT_ROOT or FORCE_RETRAIN=True')
        env=os.environ.copy()
        env.update({k:str(v) for k,v in DEFAULT_ENV.items()})
        env.update({'DATA_PATH':str(DATA_PATH),'DATA_FILE':str(DATA_PATH),
                    'OUTPUT_ROOT':str(run_dir),'REVIEWER_OUTPUT_ROOT':str(run_dir),
                    'SEED':str(SEED),'N_TIME_MODES':str(m),
                    'REVIEWER_TIME_MODES':str(m),'EPOCHS':str(EPOCHS),
                    'REVIEWER_EPOCHS':str(EPOCHS),'REVIEWER_N_TRAIN':str(n_train),
                    'WINDOW_K':str(K),'WINDOW_STRIDE':str(K),
                    'TB_PDE':PDE,'TB_N_TRAIN':str(n_train),
                    'TB_WINDOW_K':str(K),'TB_RANK_M':str(m),
                    'TB_SEED':str(SEED),'TB_TRAIN_POOL_SIZE':str(TRAIN_POOL_SIZE),
                    'TB_SOURCE_FILE':str(SOURCE_FILE),
                    'TB_RANK_OUTPUT':str(run_dir),
                    'FORCE_RETRAIN':'1' if FORCE_RETRAIN else '0',
                    'PYTHONUNBUFFERED':'1'})
        if PDE=='navier_stokes_2d':
            env.update({'NS_DATA_DIR':str(DATA_PATH.parent),'NS_OUTPUT_ROOT':str(run_dir),
                        'NS_SIZES':str(n_train),'NS_TB_RANK':str(m),
                        'NS_K':str(K),'NS_STRIDE':str(K),
                        'NS_EPOCHS':str(EPOCHS)})
        if PDE=='cahn_hilliard_2d':
            env.update({'REVIEWER_N_TRAIN':str(n_train),'REVIEWER_EPOCHS':str(EPOCHS),
                        'REVIEWER_WINDOW_K':str(K),'REVIEWER_TIME_MODES':str(m),
                        'REVIEWER_BATCH_SIZE':str(128)})
        print('\n== TRAIN',PDE,'N=',n_train,'K=',K,'M=',m,'epochs=',EPOCHS,'==',flush=True)
        with (run_dir/'run.log').open('a',encoding='utf-8') as output:
            with subprocess.Popen([sys.executable,'-u',str(RUNNER_FILE)],
                    env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                    text=True,bufsize=1,errors='replace') as proc:
                for line in proc.stdout:
                    print(line,end='',flush=True)
                    output.write(line);output.flush()
                returncode=proc.wait()
        if returncode:
            raise RuntimeError(f'Rank experiment N={n_train}, M={m} failed ({returncode}). Log: {run_dir / "run.log"}')
        assert result_path.exists(),f'Missing result after training: {result_path}'
        print('[SAVED]',result_path,flush=True)


In [ ]:
#@title Inspect all finished results (also works after interruption)
rows=[]
for n in TRAIN_SIZES:
    for m in RANKS:
        f=OUTPUT_ROOT/f'N{n}'/f'M{m}'/'result.json'
        if f.exists():
            rows.append(json.loads(f.read_text()))
if not rows:
    print('No completed experiments yet.')
else:
    df=pd.DataFrame(rows).sort_values(['N_train','M'])
    output=OUTPUT_ROOT/'tb_rank_sweep_summary.csv'
    df.to_csv(output,index=False)
    display(df[[c for c in ['N_train','K','M','epochs_completed','parameters','full_l2_mean','final_l2_mean','best_validation_window_l2','train_seconds'] if c in df.columns]])
    print('Completed',len(df),'/',len(TRAIN_SIZES)*len(RANKS),'saved to',output)
    for metric,title in [('full_l2_mean','Full-trajectory absolute L2'),('final_l2_mean','Final-time absolute L2')]:
        fig,ax=plt.subplots(figsize=(7,4))
        for m,subset in df.groupby('M'):
            ax.plot(subset['N_train'],subset[metric],marker='o',label=f'M={m}')
        ax.set(xlabel='Training trajectories',ylabel=title,title=f'{PDE}, K={K}: {title}')
        ax.legend();ax.grid(alpha=.25);fig.tight_layout()
        fig.savefig(OUTPUT_ROOT/f'{metric}_vs_N.png',dpi=160)
        plt.show()
